# Building a VLM-Backed Pipeline

In this lab, you'll explore video data that has been processed by a Vision Language Model (VLM). The VLM analyzed meeting videos using frame sampling and extracted structured segment descriptions with timestamps. You'll query these results, generate embeddings, and build semantic search queries that combine meaning and time.

Let's begin!

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> ⬇ &nbsp; <b>Download this notebook:</b> Click <em>"File"</em> → <em>"Download"</em>.</p>
</div>


## Connect to Snowflake

Load your credentials from the environment and create a Snowflake session.

In [ ]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.core import Root

_ = load_dotenv(override=True)

session = Session.builder.configs({
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "authenticator": "programmatic_access_token",
    "token": os.getenv("SNOWFLAKE_PAT"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
}).create()
root = Root(session)

DATABASE = os.getenv("SNOWFLAKE_DATABASE")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
SHARED_SCHEMA = "shared"

current_user = session.get_current_user()
LEARNER_SCHEMA = current_user.strip('"').lower()

session.use_schema(LEARNER_SCHEMA)

print(f"Connected as {current_user}")
print(f"Using schema: {DATABASE}.{LEARNER_SCHEMA}")

## Understanding the VLM Container Spec

Before you ran this lab, a Vision Language Model was deployed as a container job to process the meeting videos. Here's the container spec that was used:

```yaml
spec:
  containers:
    - name: qwen-vlm
      image: <repo_url>/qwen_vlm:latest
      resources:
        requests:
          nvidia.com/gpu: 4
        limits:
          nvidia.com/gpu: 4
      env:
        MEETING_ID: ES2008
        MEETING_PART: a
        VIDEO_PATH: /videos/amicorpus/ES2008a/video/ES2008a.Overhead.avi
        SNOWFLAKE_WAREHOUSE: DLAI_WH
        SNOWFLAKE_DATABASE: DLAI_DB
        SNOWFLAKE_SCHEMA: shared
        PROMPT: "Analyze this meeting video and identify 5-10 major segments
          or phases. For each segment, describe what is happening visually
          (e.g., presentation, group discussion, whiteboard activity) and
          what topic is being discussed. Provide 2-3 sentences of detail
          per segment. Return as a JSON array with objects containing
          start_time (hh:mm:ss), end_time (hh:mm:ss), and description
          fields."
        OUTPUT_TABLE: video_analysis
        FPS: 0.25
      volumeMounts:
        - name: videos
          mountPath: /videos
        - name: dshm
          mountPath: /dev/shm
  volumes:
    - name: dshm
      source: memory
      size: 10Gi
    - name: videos
      source: "@int_multimodal_data_files"
  networkPolicyConfig:
    allowInternetEgress: true
```

Key things to notice:

- **GPU resources**: The VLM requires 4 NVIDIA GPUs – these models are computationally expensive
- **Frame sampling** (`FPS: 0.25`): Processes 1 frame every 4 seconds, balancing cost with coverage
- **Prompt engineering**: The prompt requests structured JSON output with start/end times and descriptions
- **Shared memory** (`dshm`): A 10 GiB memory-backed volume mounted at `/dev/shm` – required for efficient GPU inter-process communication
- **Volume mounts**: The container reads video files directly from a Snowflake stage
- **Network egress**: Enabled so the container can download model weights on first run
- **Output table**: Results are written to `video_analysis`, which you’ll query next

The VLM (Qwen-VL) analyzed each video, identified 5–10 major segments per meeting part, and wrote structured results – timestamps and natural language descriptions – into the `video_analysis` table.

## Query and Analyze Video Data

Explore the extracted video segments, analyze durations, and search for specific topics in the video descriptions.

<div style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px">
<p>🔍 &nbsp; <b>Results may vary:</b> The VLM's descriptions depend on the model and frame sampling rate. Your results may differ from those shown in the video.</p>
</div>

In [ ]:
video_df = session.table(f"{SHARED_SCHEMA}.video_analysis")

print("Segment duration analysis:")
video_df.select(
    "meeting_id", "meeting_part",
    F.datediff("second", F.col("start_time"), F.col("end_time")).alias("duration_sec")
).group_by("meeting_id", "meeting_part").agg(
    F.count("*").alias("segment_count"),
    F.avg(F.col("duration_sec")).alias("avg_segment_duration_sec"),
    F.sum(F.col("duration_sec")).alias("total_video_duration_sec")
).sort("meeting_id", "meeting_part").show()

In [ ]:
print("Sample of the video segments:")
video_df.select(
    "meeting_id", "meeting_part", "start_time", "end_time",
    F.left(F.col("description"), 100).alias("description_preview")
).sort("meeting_id", "meeting_part", "start_time").limit(3).show()

In [ ]:
print("Full description for one segment:")
video_df.filter(
    (F.col("meeting_part") == "b") & (F.col("start_time") == "00:00:00")
).select("meeting_id", "meeting_part", "start_time", "end_time", "description").show(max_width=200)

Filter video segments by time window.

In [ ]:
print("Segments in first 5 minutes of each meeting:")
video_df.filter(
    F.col("start_time") < "00:05:00"
).select(
    "meeting_id", "meeting_part", "start_time", "end_time",
    F.left(F.col("description"), 80).alias("description_preview")
).sort("meeting_id", "meeting_part", "start_time").show()

## Cross-Modal Analysis

Now that you have extracted data from all three modalities — audio transcripts (Lab 2), slide content (Lab 2), and video descriptions (this lab) — you can search for a topic across all of them at once.

In [ ]:
topic = "presentation"

print(f"Searching for '{topic}' across all modalities:")
session.sql(f"""
    SELECT DISTINCT source, meeting_id, meeting_part, content FROM (
        SELECT 'AUDIO' AS source, meeting_id, meeting_part, chunk_text AS content
        FROM {LEARNER_SCHEMA}.transcript_chunks
        WHERE LOWER(chunk_text) LIKE '%{topic}%'
        LIMIT 3
    ) UNION ALL SELECT * FROM (
        SELECT 'SLIDE' AS source, meeting_id, meeting_part, extracted_text AS content
        FROM {LEARNER_SCHEMA}.slides_ocr
        WHERE LOWER(extracted_text) LIKE '%{topic}%'
        LIMIT 3
    ) UNION ALL SELECT * FROM (
        SELECT 'VIDEO' AS source, meeting_id, meeting_part, description AS content
        FROM {SHARED_SCHEMA}.video_analysis
        WHERE LOWER(description) LIKE '%{topic}%'
        LIMIT 3
    )
    ORDER BY source, meeting_id, meeting_part
""").show()

## Video Embeddings

The video descriptions already have vector embeddings generated using `snowflake-arctic-embed-m`. This is the same embedding model used for audio and slide content in Lab 2. Let's verify they exist.

In [ ]:
print("Video Embedding Samples:")

video_df = session.table(f"{SHARED_SCHEMA}.video_analysis")
video_df.filter(
    F.col("text_embedding").is_not_null()
).select(
    "text_embedding", "meeting_id", "meeting_part", "start_time", "end_time",
    F.left(F.col("description"), 80).alias("description_preview")
).limit(5).show()

## Semantic Search on Video Descriptions

Now that your video descriptions have embeddings, you can search by *meaning* rather than keywords. This finds relevant segments even when the exact words don't match your query.

In [ ]:
search_query = "product pricing discussion"

print(f"Semantic search: '{search_query}'")
session.sql(f"""
    WITH query_embedding AS (
        SELECT AI_EMBED(
            'snowflake-arctic-embed-m',
            '{search_query}'
        ) AS embedding
    )
    SELECT
        meeting_id,
        meeting_part,
        start_time,
        end_time,
        LEFT(description, 60) AS description_preview,
        ROUND(VECTOR_COSINE_SIMILARITY(text_embedding, q.embedding), 3) AS similarity
    FROM {SHARED_SCHEMA}.video_analysis, query_embedding q
    WHERE text_embedding IS NOT NULL
    ORDER BY similarity DESC
    LIMIT 5
""").show()


## Cross-Modal Semantic Search

Now search across all three modalities by meaning. This queries audio, slide, and video embeddings together – finding relevant content regardless of which modality it came from.

In [ ]:
search_query = "project budget and costs"

print(f"Cross-modal semantic search: '{search_query}'")
session.sql(f"""
    WITH query_embedding AS (
        SELECT AI_EMBED('snowflake-arctic-embed-m', '{search_query}') AS embedding
    )
    SELECT DISTINCT * FROM (
        SELECT
            'AUDIO' AS source,
            meeting_id, meeting_part,
            LEFT(chunk_text, 60) AS content_preview,
            ROUND(VECTOR_COSINE_SIMILARITY(text_embedding, q.embedding), 3) AS similarity
        FROM {LEARNER_SCHEMA}.transcript_chunks, query_embedding q
        WHERE text_embedding IS NOT NULL

        UNION ALL

        SELECT
            'SLIDE' AS source,
            meeting_id, meeting_part,
            LEFT(extracted_text, 60) AS content_preview,
            ROUND(VECTOR_COSINE_SIMILARITY(text_embedding, q.embedding), 3) AS similarity
        FROM {LEARNER_SCHEMA}.slides_ocr, query_embedding q
        WHERE text_embedding IS NOT NULL

        UNION ALL

        SELECT
            'VIDEO' AS source,
            meeting_id, meeting_part,
            LEFT(description, 60) AS content_preview,
            ROUND(VECTOR_COSINE_SIMILARITY(text_embedding, q.embedding), 3) AS similarity
        FROM {SHARED_SCHEMA}.video_analysis, query_embedding q
        WHERE text_embedding IS NOT NULL
    )
    ORDER BY similarity DESC
    LIMIT 10
""").show()


## Combined Time + Semantic Query

Filter by both time window and semantic similarity. This answers questions like "find segments about design in the first 3 minutes."

In [ ]:
search_query = "design discussion"

print(f"Segments about '{search_query}' in the first 3 minutes:")
session.sql(f"""
    SELECT
        meeting_id,
        meeting_part,
        start_time,
        end_time,
        LEFT(description, 80) AS description_preview,
        ROUND(similarity, 3) AS similarity
    FROM (
        SELECT
            *,
            VECTOR_COSINE_SIMILARITY(
                text_embedding,
                AI_EMBED('snowflake-arctic-embed-m', '{search_query}')
            ) AS similarity
        FROM {SHARED_SCHEMA}.video_analysis
        WHERE text_embedding IS NOT NULL
    )
    WHERE start_time < '00:03:00'
      AND similarity > 0.3
    ORDER BY similarity DESC
    LIMIT 5
""").show()

## Time-Based Aggregation

Use semantic similarity to answer questions about *how much time* was spent on a topic. This sums up the duration of all segments that match a semantic query.

In [ ]:
search_query = "product pricing and budget"

print(f"Time spent on '{search_query}' topics:")
session.sql(f"""
    SELECT
        ROUND(SUM(TIMESTAMPDIFF(SECOND, start_time, end_time)) / 60.0, 1) AS total_minutes,
        COUNT(*) AS segment_count
    FROM (
        SELECT
            *,
            VECTOR_COSINE_SIMILARITY(
                text_embedding,
                AI_EMBED('snowflake-arctic-embed-m', '{search_query}')
            ) AS similarity
        FROM {SHARED_SCHEMA}.video_analysis
        WHERE text_embedding IS NOT NULL
    )
    WHERE similarity > 0.7
""").show()

## Next Steps

In this lab, you:

1. **Reviewed the VLM container spec** — Understood how the VLM was deployed with frame sampling, prompt engineering, and structured output
2. **Explored video analysis results** — Queried segments with timestamps and analyzed durations
3. **Cross-modal keyword search** — Searched across audio, slides, and video for topics
4. **Generated video embeddings** — Created vector embeddings for semantic search
5. **Semantic search** — Found video segments by meaning, not just keywords
6. **Time + semantic queries** — Combined time windows with semantic similarity
7. **Time-based aggregations** — Measured how much time was spent on topics

All three modalities now have vector embeddings for semantic search:
- Audio transcript chunks (`transcript_chunks.text_embedding`)
- Slide OCR text (`slides_ocr.text_embedding`)
- Video descriptions (`video_analysis.text_embedding`)

In the next lab, you'll build a **RAG pipeline** that queries across all three modalities to answer questions about the meetings.

## References 

- **Defining a container spec in Snowflake**: The VLM shown in this lab is deployed as a Snowpark Container Services job. The full YAML schema (containers, resources, `env`, `volumeMounts`, `volumes`, `networkPolicyConfig`, etc.) is documented in Snowflake's [Service specification reference](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/specification-reference). Use it as a guide when adapting the spec above to your own VLM or GPU workload.